# 02 - OCR-supported training and evaluation

Run this workflow's baseline first, then restart the kernel. Train a fresh VLM LoRA on 72 transcript targets and compare both pipelines on the same eight validation sequences. The second text formatting model remains frozen with identical settings.

Run the cells in order from a fresh kernel. The VLM generates and saves all validation transcripts first; it is then unloaded before the frozen text formatter loads. Raw transcripts are kept in the evaluation output folder under `raw_transcripts/predictions.jsonl`.

To retry formatting in a fresh kernel without regenerating images, run only the setup and dataset cells, then the **Stage 2** cells below. Do not run the VLM loading or training cells for that retry.


In [ ]:
import gc
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from peft import LoraConfig
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

# Find the project folder when running from the root or notebooks folder.
ROOT = Path.cwd().resolve()
if not (ROOT / "Scripts").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "Scripts" / "trainer.py").is_file():
    raise FileNotFoundError("Run this notebook from the project or notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Scripts.dataUtils import (
    MangaCollator,
    load_labels,
)
from Scripts.evaluator import (
    calculate_metrics,
    display_metric_summary,
    evaluate_model,
    inspect_prediction,
    load_prediction_records,
)
from Scripts.trainer import build_lora_model, build_trainer, build_training_args

SEED = 42
MODEL_ROOT = Path(r"C:\MODELS\HF")
MODEL_PATH = Path(os.environ.get("MANGA_VLM_MODEL_PATH") or MODEL_ROOT / "Qwen2.5-VL-3B-Instruct").expanduser()
FORMATTER_MODEL_PATH = Path(os.environ.get("MANGA_FORMATTER_MODEL_PATH") or MODEL_ROOT / "Qwen2.5-1.5B-Instruct").expanduser()
for model_role, model_path, override_name in (
    ("VLM", MODEL_PATH, "MANGA_VLM_MODEL_PATH"),
    ("JSON formatter", FORMATTER_MODEL_PATH, "MANGA_FORMATTER_MODEL_PATH"),
): 
    if not model_path.is_dir():
        raise FileNotFoundError(f"{model_role} model folder not found: {model_path}. Download the model there or set {override_name} to its local folder.")
    print(f"{model_role} model path: {model_path}")
LABELS_PATH = ROOT / "TaskData" / "development" / "labels.jsonl"
IMAGE_ROOT = ROOT / "TaskData" / "development" / "images"
SPLIT_PATH = ROOT / "outputs" / "canonical_split_seed42.json"
# Match OUTPUT_DIR in 01_Baseline.ipynb.
BASELINE_DIR = ROOT / "outputs" / "ocr_base_baseline"
RUN_DIR = ROOT / "artifacts" / "ocr_lora"
EVAL_DIR = ROOT / "outputs" / "ocr_lora_eval"
MAX_NEW_TOKENS = 1024
MINIMUM_SIMILARITY = 0.50
REPETITION_PENALTY = 1.05

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from Scripts.modelFormatter import (FormatterSequenceDataset, FormatterPromptBuilder, ModelJSONFormatter, export_transcript_dataset, format_transcript, format_saved_transcripts)
WORKFLOW_DIR = ROOT / "notebooks-OCRsupported"
if str(WORKFLOW_DIR) not in sys.path:
    sys.path.insert(0, str(WORKFLOW_DIR))
from ocr_support import OCRPromptBuilder, prepare_ocr
OCR_LANGUAGES = ["en"]
OCR_CACHE_DIR = WORKFLOW_DIR / "datasets" / "ocr_cache"

TRANSCRIPT_PROMPT = (WORKFLOW_DIR / "prompts" / "transcript.txt").read_text(encoding="utf-8").strip()
FORMATTER_PROMPT = (WORKFLOW_DIR / "prompts" / "formatter.txt").read_text(encoding="utf-8").strip()
FORMATTER_DEVICE = os.environ.get("MANGA_FORMATTER_DEVICE", "cpu")
FORMATTER_MAX_NEW_TOKENS = 2048

RAW_TRANSCRIPT_DIR = EVAL_DIR / "raw_transcripts"
TRANSCRIPT_PATH = RAW_TRANSCRIPT_DIR / "predictions.jsonl"


In [ ]:
if not SPLIT_PATH.is_file():
    raise FileNotFoundError("Run 01_Baseline.ipynb first to save the split")
if not (BASELINE_DIR / "predictions.jsonl").is_file():
    raise FileNotFoundError(f"Run the baseline evaluation first: {BASELINE_DIR}")

split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
labels = load_labels(LABELS_PATH)
train_ids = split["train_ids"]
validation_ids = split["validation_ids"]
assert set(train_ids).isdisjoint(validation_ids)
assert set(train_ids) | set(validation_ids) == set(labels)

train_dataset = FormatterSequenceDataset(LABELS_PATH, IMAGE_ROOT, train_ids)
validation_dataset = FormatterSequenceDataset(LABELS_PATH, IMAGE_ROOT, validation_ids)
print(f"Training: {len(train_dataset)}, validation: {len(validation_dataset)}")
workflow_split = json.loads((WORKFLOW_DIR / "datasets" / "split_seed42.json").read_text(encoding="utf-8"))
if workflow_split != split:
    raise RuntimeError("Workflow dataset split differs from the canonical split; rerun the baseline")
assert len(train_dataset) == 72 and len(validation_dataset) == 8
export_transcript_dataset(train_dataset, WORKFLOW_DIR / "datasets" / "train.jsonl")
export_transcript_dataset(validation_dataset, WORKFLOW_DIR / "datasets" / "validation.jsonl")


In [ ]:
# OCR sees only source images, never reference text or speaker labels.
# Run before loading the VLM; EasyOCR uses CPU and cached full-resolution pages.
ocr_hints = prepare_ocr(train_dataset, OCR_CACHE_DIR, OCR_LANGUAGES)
ocr_hints.update(prepare_ocr(validation_dataset, OCR_CACHE_DIR, OCR_LANGUAGES))
sample = validation_dataset[0]
for page, image_path in enumerate(sample["image_paths"], 1):
    print(f"OCR PAGE {page}: {image_path.name}")
    display(pd.DataFrame(ocr_hints[str(image_path.resolve())], columns=["box", "text", "confidence"]))


In [ ]:
# Release a formatter left from an earlier run before loading the VLM.
globals().pop("formatter", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for 4-bit LoRA training")
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Local model not found: {MODEL_PATH}")

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map={"": 0},
)

prompt_builder = OCRPromptBuilder(TRANSCRIPT_PROMPT, ocr_hints)
train_collator = MangaCollator(
    processor, process_vision_info, prompt_builder, train_mode=True,
)
eval_collator = MangaCollator(
    processor, process_vision_info, prompt_builder,
    train_mode=False, return_metadata=True,
)



experiment = {"pipeline": "images + OCR hints -> VLM transcript -> text model -> JSON", "ocr": {"engine": "easyocr", "languages": OCR_LANGUAGES, "gpu": False, "detail": 1, "paragraph": False, "cache_version": 1}, "vlm_model": str(MODEL_PATH), "formatter_model": str(FORMATTER_MODEL_PATH), "formatter_device": FORMATTER_DEVICE, "transcript_prompt": TRANSCRIPT_PROMPT, "formatter_prompt": FORMATTER_PROMPT, "split": split, "max_new_tokens": MAX_NEW_TOKENS, "formatter_max_new_tokens": FORMATTER_MAX_NEW_TOKENS, "repetition_penalty": REPETITION_PENALTY, "min_pixels": 128 * 28 * 28, "max_pixels": 256 * 28 * 28}
baseline_experiment = json.loads((BASELINE_DIR / "experiment.json").read_text(encoding="utf-8"))
if baseline_experiment != experiment:
    raise RuntimeError("Baseline configuration differs; rerun the OCR-supported baseline with these settings")
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = build_lora_model(base_model, lora_config)
model.print_trainable_parameters()

# The helper uses batch size 1, gradient accumulation 8, and learning rate 1e-4.
# It evaluates and saves each epoch, then loads the model with the lowest loss.
training_args = build_training_args(
    output_dir=str(RUN_DIR),
    num_train_epochs=5,
    logging_steps=5,
    warmup_steps=5,
)
trainer = build_trainer(
    model, train_dataset, train_collator, training_args, validation_dataset,
)

In [ ]:
train_result = trainer.train()
display(pd.Series(train_result.metrics, name="training"))

FINAL_DIR = RUN_DIR / "final"
trainer.save_model(str(FINAL_DIR))
processor.save_pretrained(str(FINAL_DIR))
print(f"Saved best adapter to {FINAL_DIR}")

In [ ]:
# Stage 1: generate every VLM transcript without loading the formatter.
evaluate_model(
    model=model,
    processor=processor,
    dataset=validation_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=MAX_NEW_TOKENS,
    minimum_similarity=MINIMUM_SIMILARITY,
    repetition_penalty=REPETITION_PENALTY,
    output_dir=RAW_TRANSCRIPT_DIR,
    generation_only=True,
)
print(f"Saved raw VLM transcripts to {TRANSCRIPT_PATH}")


## Stage 2: unload the VLM and format saved transcripts

This cell reads the saved transcripts from disk. The formatter receives only transcript text; targets and images remain evaluation metadata. Training state is also released in the LoRA notebook. Formatting failures remain in the scored outputs.


In [ ]:
# Confirm all holdout transcripts are saved before releasing the VLM.
saved_transcripts = load_prediction_records(TRANSCRIPT_PATH)
if sorted(row["sequence_id"] for row in saved_transcripts) != sorted(validation_ids):
    raise RuntimeError("Saved transcripts do not match the complete validation split; rerun Stage 1")
del saved_transcripts

# Drop every notebook reference to the VLM and training state before formatting.
for name in ("trainer", "model", "base_model", "processor", "train_collator",
             "eval_collator", "sample_inputs", "sample_metadata", "generated",
             "formatter"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VLM and training state released; loading the text formatter next.")

formatter = ModelJSONFormatter(
    FORMATTER_MODEL_PATH, FORMATTER_PROMPT,
    device=FORMATTER_DEVICE, max_new_tokens=FORMATTER_MAX_NEW_TOKENS,
)
try:
    tuned_metrics = format_saved_transcripts(
        TRANSCRIPT_PATH, formatter, EVAL_DIR,
        minimum_similarity=MINIMUM_SIMILARITY,
    )
finally:
    del formatter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"Saved formatted JSON predictions and metrics to {EVAL_DIR}")


Compare the model formatter with a tolerant deterministic parser on the same transcripts. The model remains the main pipeline; parser predictions and metrics are saved separately.


In [ ]:
# Compare serialization on the same saved VLM transcripts; no model reload.
parser_metrics = format_saved_transcripts(
    TRANSCRIPT_PATH, lambda text: format_transcript(text, tolerant=True),
    EVAL_DIR / "tolerant_parser", minimum_similarity=MINIMUM_SIMILARITY,
)
display(pd.DataFrame([
    {"formatter": "text model", "schema_compliance_rate": tuned_metrics["schema_compliance_rate"]},
    {"formatter": "tolerant parser", "schema_compliance_rate": parser_metrics["schema_compliance_rate"]},
]))
# Full task metrics and per-sequence failures are saved separately under tolerant_parser.


In [ ]:
baseline_records = load_prediction_records(BASELINE_DIR / "predictions.jsonl")
assert sorted(record["sequence_id"] for record in baseline_records) == sorted(validation_ids)
baseline_metrics = calculate_metrics(
    baseline_records, minimum_similarity=MINIMUM_SIMILARITY,
)
metric_names = [
    "cer_valid_outputs",
    "cer_including_failures_as_deletions",
    "cer_substitution_rate",
    "cer_deletion_rate",
    "cer_insertion_rate",
    "dialogue_text_match_f1",
    "matched_speaker_accuracy",
    "b3_f1",
    "identity_switch_rate",
    "narration_f1",
    "json_parse_rate",
    "schema_compliance_rate",
    "sequence_exact_match",
]
comparison = pd.DataFrame({
    "base_model": [baseline_metrics[name] for name in metric_names],
    "tuned": [tuned_metrics[name] for name in metric_names],
}, index=metric_names)
comparison["delta"] = comparison["tuned"] - comparison["base_model"]
display(comparison.round(4))

In [ ]:
records = load_prediction_records(EVAL_DIR / "predictions.jsonl")
rows = []
for record in records:
    metrics = calculate_metrics([record], minimum_similarity=MINIMUM_SIMILARITY)
    rows.append({
        "sequence_id": record["sequence_id"],
        "format_valid": bool(metrics["schema_compliance_rate"]),
        "text_error_rate": metrics["cer_valid_outputs"],
        "failure_penalty_cer": metrics["cer_including_failures_as_deletions"],
        "speaker_accuracy": metrics["matched_speaker_accuracy"],
        "dialogue_f1": metrics["dialogue_text_match_f1"],
        "generated_tokens": record["generated_tokens"],
        "hit_max_new_tokens": record["hit_max_new_tokens"],
        "repetition_loop": record["suspected_repetition_loop"],
    })

review = pd.DataFrame(rows).sort_values(
    ["format_valid", "failure_penalty_cer"], ascending=[True, False],
)
display(review.round(4))

In [ ]:
# Inspect the sequence with the worst result.
_ = inspect_prediction(
    EVAL_DIR / "predictions.jsonl",
    sequence_id=review.iloc[0]["sequence_id"],
    minimum_similarity=MINIMUM_SIMILARITY,
)

## Official task score: primary experiment benchmark

Use the unmodified `TaskData/score.py` to judge text order and balanced text/speaker quality on the fixed holdout. Inspect invalid outputs and missing sequences before accepting an improvement.


In [ ]:
from Scripts.taskScore import score_saved_predictions

official_base = score_saved_predictions(BASELINE_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, BASELINE_DIR / "task_score")
official_tuned = score_saved_predictions(EVAL_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, EVAL_DIR / "task_score")
official_comparison = pd.DataFrame({"base_model": official_base["macro"], "tuned": official_tuned["macro"]})
official_comparison["delta"] = official_comparison["tuned"] - official_comparison["base_model"]
display(official_comparison.round(4))
print("Missing or invalid tuned sequences:", official_tuned["missing_sequences"])
